# 01 数据获取与观测准备

这个 notebook 是目标获取和观测准备的稳定入口。它汇总 `README.md` 里的 TNS、Lasair、WISeREP 流程，并把远程下载设为手动开启，方便汇报时安全打开。

科学作用：在进入光谱分析前，先整理候选体元数据、可观测窗口、找星图、公开光谱和光变曲线。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from importlib import reload

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import spectral_pipeline as sp
from src import spectral_notebook_tools as snt

OUTPUT_DIR = PROJECT_ROOT / "output"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_pipeline"
FIG_DIR = ANALYSIS_DIR / "figures"
CONFIG_PATH = PROJECT_ROOT / "configs" / "sn_parameter.json"

## 配置快照

主观测流水线读取 `configs/sn_parameter.json`。TNS 和 Lasair 凭证从 `.env` 读取；`.env` 不提交到 git。

In [ ]:
config = json.loads(CONFIG_PATH.read_text())
display(config)

## 可选：刷新远程数据

平时写报告时保持 `RUN_REMOTE_FETCH = False`。只有需要重新拉取 TNS、找星图、Lasair 光变曲线和 WISeREP 光谱时，才把它改成 `True`。

In [ ]:
RUN_REMOTE_FETCH = False

if RUN_REMOTE_FETCH:
    subprocess.run([sys.executable, "scripts/fetch_target_params.py"], cwd=PROJECT_ROOT, check=True)
    subprocess.run([sys.executable, "scripts/fetch_aux_data.py"], cwd=PROJECT_ROOT, check=True)
else:
    print("跳过远程刷新，使用现有 output/ 和 data/ 产物。")

## 已有目标产物盘点

下面统计已经生成的观测报告、找星图、光变曲线、光谱和模型输出。

In [ ]:
products = []
for path in sorted(OUTPUT_DIR.glob("*")):
    if not path.is_dir() or path.name == "analysis_pipeline":
        continue
    products.append({
        "target": path.name,
        "reports": len(list(path.glob("sn_report_*.txt"))),
        "finder_charts": len(list(path.glob("finder_*"))),
        "lightcurve_files": len(list((path / "lightcurve").glob("*"))) if (path / "lightcurve").exists() else 0,
        "spectra_files": len(list((path / "spectrum").glob("*"))) if (path / "spectrum").exists() else 0,
        "superfit_files": len(list((path / "superfit").glob("*"))) if (path / "superfit").exists() else 0,
        "tardis_files": len(list((path / "tardis").glob("*"))) if (path / "tardis").exists() else 0,
    })
products_df = pd.DataFrame(products)
display(products_df)

## 可选：重新运行 Superfit

默认保持 `RUN_SUPERFIT = False`，避免打开 notebook 就批量重跑。需要刷新模板拟合时，先在 `SUPERFIT_TARGETS` 里指定目标；空列表表示对所有 `data/SN*/SN*.txt` 重跑。若有可靠宿主红移，可填入 `SUPERFIT_Z_BY_TARGET`，否则 Superfit 会在粗红移网格中搜索。

In [ ]:
RUN_SUPERFIT = True
SUPERFIT_TARGETS = []  # 例如 ["SN2026KID"]; 空列表表示全部本地 txt 光谱
SUPERFIT_Z_BY_TARGET = {
    # "SN2026KID": 0.0014,
}

if RUN_SUPERFIT:
    superfit_run_table = snt.run_superfit_batch(
        PROJECT_ROOT,
        targets=SUPERFIT_TARGETS,
        z_by_target=SUPERFIT_Z_BY_TARGET,
        z_range=(0.0, 0.08),
        z_step=0.005,
        resolution=30,
        how_many_plots=5,
    )
    display(superfit_run_table)
else:
    print("未重跑 Superfit；使用已有 data/SN*/superfit/*.csv。")

## 可选：重新运行 DASH

默认保持 `RUN_DASH = False`。DASH 会读取 `data/SN*/SN*.txt`，并把匹配结果写回 `notebooks/DASH_matches.txt`。如果 `DASH_KNOWN_Z = True`，需要为所有待跑目标在 `DASH_Z_BY_TARGET` 填可靠红移；否则 DASH 会自己估计红移。

In [ ]:
RUN_DASH = False
DASH_TARGETS = []  # 例如 ["SN2026JLM"]; 空列表表示全部本地 txt 光谱
DASH_KNOWN_Z = False
DASH_Z_BY_TARGET = {
    # "SN2026JLM": 0.0155,
}

if RUN_DASH:
    dash_run_table = snt.run_dash_batch(
        PROJECT_ROOT,
        targets=DASH_TARGETS,
        z_by_target=DASH_Z_BY_TARGET,
        known_z=DASH_KNOWN_Z,
        output_path=PROJECT_ROOT / "notebooks" / "DASH_matches.txt",
        top_n=5,
    )
    display(dash_run_table)
else:
    print("未重跑 DASH；使用已有 notebooks/DASH_matches.txt。")

## 已有模板工具产物：Superfit/DASH 状态

这里盘点 `data/SN*/superfit/*.csv` 和 `notebooks/DASH_matches.txt` 里已有或刚重跑出的模板分类结果，作为后面经验粗分类的交叉参考。

In [ ]:
superfit_summary = snt.summarize_existing_superfit_results(PROJECT_ROOT)
if superfit_summary.empty:
    print("没有发现已有 Superfit CSV。")
else:
    display(superfit_summary)

template_spectrum_table, template_target_table = snt.summarize_local_template_classifications(PROJECT_ROOT)
if template_target_table.empty:
    print("没有可汇总的本地模板分类结果。")
else:
    display(template_target_table)

dash_log = PROJECT_ROOT / "notebooks" / "DASH_matches.txt"
if dash_log.exists():
    print(f"已有 DASH 文本记录：{dash_log}")
else:
    print("没有发现 DASH_matches.txt。")

## 全自动本地光谱粗分类

这一步只读取 `data/` 下自己的 FITS 光谱，不读 `data/tns_public_objects.csv`，也不读 TNS 类型。方法是轻量级经验分类：先用宿主窄发射线粗估红移，再测 Si、H、He、O、Ca、Fe 等关键谱线的吸收特征，给出 Ia/II/IIb/Ib/Ic 的初筛类型、粗红移、代表速度和黑体颜色温度。

这不是 DASH/SNID/Superfit 模板匹配，只用于观测准备和 02 notebook 自动选线；最终报告中的类型仍应结合人工检查或模板工具确认。

In [ ]:
reload(snt)

spectra_local, skipped_local = snt.load_observed_spectra(PROJECT_ROOT, target_metadata={})
if not spectra_local:
    print("没有找到可处理的一维 FITS 光谱。")
else:
    rough_spectrum_table, rough_target_table, rough_feature_table = snt.rough_classify_spectra(spectra_local)
    template_spectrum_table, template_target_table = snt.summarize_local_template_classifications(PROJECT_ROOT)
    classification_target_table = snt.combine_template_and_rough_classifications(template_target_table, rough_target_table)

    display(classification_target_table)
    display(rough_target_table)
    display(rough_spectrum_table[[
        "target", "file", "date_obs", "rough_type", "rough_type_confidence",
        "rough_z", "rough_z_source", "rough_velocity_line", "rough_velocity_kms", "T_bb_K",
    ]])
    snt.plot_rough_classification_summary(classification_target_table, FIG_DIR, save_figures=True)

    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    classification_target_table.to_csv(ANALYSIS_DIR / "local_classification_targets.csv", index=False)
    template_spectrum_table.to_csv(ANALYSIS_DIR / "local_template_classification_spectra.csv", index=False)
    template_target_table.to_csv(ANALYSIS_DIR / "local_template_classification_targets.csv", index=False)
    rough_target_table.to_csv(ANALYSIS_DIR / "rough_classification_targets.csv", index=False)
    rough_spectrum_table.to_csv(ANALYSIS_DIR / "rough_classification_spectra.csv", index=False)
    rough_feature_table.to_csv(ANALYSIS_DIR / "rough_classification_line_features.csv", index=False)
    skipped_local.to_csv(ANALYSIS_DIR / "rough_classification_skipped_fits.csv", index=False)
    print(f"Saved rough-classification products to {ANALYSIS_DIR}")

## 光谱分析 pipeline 给出的目标状态

运行 `02_spectral_analysis_pipeline.ipynb` 后，单目标调参输出会带目标名前缀；运行 `scripts/build_analysis_products.py` 后可能还有旧的全量无前缀表。这里按目标合并读取最新可用版本。

In [ ]:
status_table = snt.read_combined_analysis_products(ANALYSIS_DIR, "target_status.csv")
status_products = snt.find_analysis_products(ANALYSIS_DIR, "target_status.csv")

if status_table.empty:
    print(f"缺少 target_status.csv 或 *_target_status.csv。请先运行光谱分析 pipeline。")
else:
    print("读取到的目标状态产物（新到旧）：")
    for path in status_products[:8]:
        print(f"- {path.name}")
    display(status_table)